# Titanic Survival Prediction - Complete Guide

Welcome to this comprehensive guide on solving the Titanic survival prediction problem. This notebook is designed to explain each step of the Machine Learning pipeline, from data exploration to model evaluation.


## 1. Importing Libraries
Let's start by importing the necessary libraries for data manipulation, visualization, and machine learning.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

# Set style for plots
sns.set_theme(style='whitegrid')
import warnings
warnings.filterwarnings('ignore')


## 2. Loading the Dataset
For this guide, we will use the Titanic dataset built into the Seaborn library. In a real Kaggle competition, you would load `train.csv` and `test.csv` using `pd.read_csv()`. Here, seaborn provides a slightly pre-processed version of the Titanic dataset, which is great for learning.


In [ ]:
# Load the titanic dataset from seaborn
df = sns.load_dataset('titanic')

# Display the first few rows
df.head()


## 3. Exploratory Data Analysis (EDA)
Let's understand the data and find relationships between features and the target variable (`survived`).


In [ ]:
# Check the dataset info to see columns, non-null counts, and data types
df.info()


In [ ]:
# Summary statistics for numerical columns
df.describe()


### Survival Rate by Gender
Let's see how the survival rate differs between males and females.


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='survived', hue='sex')
plt.title('Survival by Gender')
plt.show()


### Survival Rate by Passenger Class (Pclass)
Passenger class (1st, 2nd, 3rd) often correlates strongly with survival.


In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x='survived', hue='pclass')
plt.title('Survival by Passenger Class')
plt.show()


## 4. Data Preprocessing & Cleaning
Real-world data is messy. We need to handle missing values and drop unnecessary columns. Let's see what's missing.


In [ ]:
# Check for missing values
df.isnull().sum()


We have missing values in `age`, `deck`, and `embarked`. Let's handle them:
- **age**: We will fill this with the median age.
- **embarked**: We will fill this with the most frequent value (mode).
- **deck**: This has too many missing values, so we'll drop this column.
- We'll also drop redundant columns like `alive` (same as survived), `class` (same as pclass), `who`, `adult_male`, and `embark_town`.


In [ ]:
# Fill missing age with median
df['age'].fillna(df['age'].median(), inplace=True)

# Fill missing embarked with mode
df['embarked'].fillna(df['embarked'].mode()[0], inplace=True)

# Drop columns that are redundant or have too many missing values
cols_to_drop = ['deck', 'alive', 'class', 'who', 'adult_male', 'embark_town']
df.drop(columns=cols_to_drop, inplace=True)

print('Missing values after cleaning:')
print(df.isnull().sum())


## 5. Feature Engineering
Let's create new features that might help our model capture patterns more effectively.


In [ ]:
# Create a FamilySize feature (sibsp = siblings/spouses, parch = parents/children)
# Adding 1 to include the passenger themselves
df['FamilySize'] = df['sibsp'] + df['parch'] + 1

# Create an IsAlone binary feature
df['IsAlone'] = 0
df.loc[df['FamilySize'] == 1, 'IsAlone'] = 1

df[['FamilySize', 'IsAlone']].head()


Next, we convert categorical variables (text) into numerical formats so the machine learning model can process them.


In [ ]:
# Convert 'sex' to binary (0 for male, 1 for female)
df['sex'] = df['sex'].map({'male': 0, 'female': 1})

# One-Hot Encode 'embarked' (C = Cherbourg, Q = Queenstown, S = Southampton)
# drop_first=True helps avoid the dummy variable trap
df = pd.get_dummies(df, columns=['embarked'], drop_first=True)

df.head()


## 6. Train-Test Split & Scaling
We split our data into a training set (to train the model) and a testing set (to evaluate the model's performance on unseen data).


In [ ]:
# Separate features (X) and target (y)
X = df.drop('survived', axis=1)
y = df['survived']

# Split the data (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Feature Scaling (Standardization)
# This ensures that all features have a mean of 0 and standard deviation of 1
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print(f'Training data shape: {X_train.shape}')
print(f'Testing data shape: {X_test.shape}')


## 7. Model Building
We will train two models: Logistic Regression (a solid baseline) and Random Forest (a powerful ensemble model).


In [ ]:
### 1. Logistic Regression
log_reg = LogisticRegression()
log_reg.fit(X_train, y_train)

y_pred_log = log_reg.predict(X_test)
print('Logistic Regression Accuracy:', accuracy_score(y_test, y_pred_log))


In [ ]:
### 2. Random Forest Classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
print('Random Forest Accuracy:', accuracy_score(y_test, y_pred_rf))


## 8. Model Evaluation
Let's evaluate the Random Forest model in more detail using a confusion matrix and classification report.


In [ ]:
print('Classification Report (Random Forest):\n')
print(classification_report(y_test, y_pred_rf))


In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred_rf)
plt.figure(figsize=(5,4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix - Random Forest')
plt.show()


### Feature Importance
Which features were the most important for predicting survival according to our Random Forest model?


In [ ]:
feature_importances = pd.Series(rf_model.feature_importances_, index=X.columns)
feature_importances.nlargest(10).plot(kind='barh', figsize=(8, 5))
plt.title('Feature Importances')
plt.xlabel('Importance Score')
plt.ylabel('Feature')
plt.show()


## 9. Conclusion
In this notebook, we successfully built an end-to-end Machine Learning pipeline to predict Titanic survival:
1. **Explored** the data to understand the role of Gender and Passenger Class in survival rates.
2. **Cleaned** the data by handling missing values (Age, Embarked) and removing irrelevant columns.
3. **Engineered** new features like `FamilySize` and `IsAlone` to help the model learn better.
4. **Prepared** the data using encoding and scaling.
5. **Trained** Logistic Regression and Random Forest models, achieving strong predictive accuracy (~80-82%).
6. **Evaluated** the model and found out that `Sex` (Gender), `Age`, and `Fare` were the most important factors for survival.

*Note: To submit to a Kaggle competition, you would apply the exact same preprocessing steps to the `test.csv` file and output a CSV file with `PassengerId` and `Survived` predictions.*
